In [1]:
import sys
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
sys.path.append(os.path.abspath(".."))
import config



In [2]:
train_dir = "../dataset_train/"
test_dir = "../dataset_test/"

In [3]:
img_width, img_height = config.IMG_SIZE
batch_size = 32

train_fraction = 0.05  
test_fraction = 0.05

In [4]:
def create_dataframe(root_dir, fraction=1.0):
    """
    Create a DataFrame with two columns:
      - 'filename': path relative to root_dir of the image (e.g., "animals/fake/img1.jpg")
      - 'class': label extracted from the leaf folder (e.g., "fake" or "real")
      
    Only uses a fraction of the images if desired.
    """
    records = []
    for category in os.listdir(root_dir):
        cat_path = os.path.join(root_dir, category)
        if not os.path.isdir(cat_path):
            continue
        for subcat in os.listdir(cat_path):
            subcat_path = os.path.join(cat_path, subcat)
            if not os.path.isdir(subcat_path):
                continue
            for fname in os.listdir(subcat_path):
                file_path = os.path.join(subcat_path, fname)
                if os.path.isfile(file_path):
                    # Save the path relative to root_dir and use subcat (fake/real) as the label
                    rel_path = os.path.join(category, subcat, fname)
                    records.append({'filename': rel_path, 'class': subcat})
    df = pd.DataFrame(records)
    if fraction < 1.0:
        df = df.sample(frac=fraction, random_state=42).reset_index(drop=True)
    return df

In [5]:
train_df = create_dataframe(train_dir, fraction=train_fraction)
test_df = create_dataframe(test_dir, fraction=test_fraction)

print(f"Using {len(train_df)} training images and {len(test_df)} testing images.")

Using 15117 training images and 3779 testing images.


In [6]:
datagen = ImageDataGenerator(rescale=1./255)

In [7]:
train_generator = datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=train_dir,      
    x_col="filename",
    y_col="class",
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical' 
)

validation_generator = datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=test_dir,
    x_col="filename",
    y_col="class",
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode='categorical'
)

Found 15117 validated image filenames belonging to 2 classes.
Found 3779 validated image filenames belonging to 2 classes.


In [8]:
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu',
                  input_shape=(img_width, img_height, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(2, activation='softmax')  # fake and real
])

model.compile(optimizer=optimizers.Adam(),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

/Users/eliotatlani/micromamba/envs/cs109b/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 198, 198, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 99, 99, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 97, 97, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 48, 48, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 46, 46, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 23, 23, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 67712)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     8,667,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,760,770 (33.42 MB)

 Trainable params: 8,760,770 (33.42 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs = 5

# early stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator,
    callbacks=[early_stopping]
)

Epoch 1/5


/Users/eliotatlani/micromamba/envs/cs109b/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


473/473 ━━━━━━━━━━━━━━━━━━━━ 167s 351ms/step - accuracy: 0.5293 - loss: 0.7357 - val_accuracy: 0.5348 - val_loss: 0.6908
Epoch 2/5
473/473 ━━━━━━━━━━━━━━━━━━━━ 165s 349ms/step - accuracy: 0.5407 - loss: 0.6902 - val_accuracy: 0.5348 - val_loss: 0.6908
Epoch 3/5
473/473 ━━━━━━━━━━━━━━━━━━━━ 172s 364ms/step - accuracy: 0.5381 - loss: 0.6916 - val_accuracy: 0.5377 - val_loss: 0.6894
Epoch 4/5
270/473 ━━━━━━━━━━━━━━━━━━━━ 1:05 323ms/step - accuracy: 0.5493 - loss: 0.6889

In [ ]:
# save the model
model.save('../weights/baseline_model.h5')

In [ ]:
loss, accuracy = model.evaluate(validation_generator)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

In [ ]:
# plot the training history
import matplotlib.pyplot as plt
def plot_history(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs = range(len(acc))

    plt.plot(epochs, acc, 'b', label='Training accuracy')
    plt.plot(epochs, val_acc, 'r', label='Validation accuracy')
    plt.title('Training and validation accuracy')
    plt.legend()

    plt.figure()

    plt.plot(epochs, loss, 'b', label='Training loss')
    plt.plot(epochs, val_loss, 'r', label='Validation loss')
    plt.title('Training and validation loss')
    plt.legend()

    plt.show()
    
plot_history(history)

